# 🔴 PIXEL Mars Rover — GRPO Training with Meta Llama 3.1 8B

Train a small LLM to control a Mars rover using **Group Relative Policy Optimization (GRPO)**.

| Component | Value |
|-----------|-------|
| Model | unsloth/meta-llama-3.1-8b-instruct (4-bit LoRA via Unsloth) |
| Framework | HuggingFace TRL GRPOTrainer |
| Environment | PIXEL OpenEnv Mars Rover |
| Reward | Composable rubric (science + survival + efficiency + coordination) |

## Section 1 — Install Dependencies

In [ ]:
!pip install -q "unsloth[colab-new]" trl transformers datasets
!pip install -q torch wandb matplotlib numpy
!pip install -q openenv-core fastapi uvicorn pydantic

# Upload or clone your PIXEL environment
# !git clone https://github.com/your-username/PIXIE.git
# Or upload the pixel/ folder manually

## Section 2 — Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/meta-llama-3.1-8b-instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print(f"Loaded {MODEL_NAME} with LoRA adapters")

## Section 3 — Environment & Prompt Dataset

In [ ]:
import sys, os, random, re
sys.path.insert(0, '.')  # adjust if pixel/ is elsewhere

from backend.environment import PIXELEnvironment, parse_action
from backend.rewards import EpisodeTracker
from datasets import Dataset

SYSTEM_PROMPT = """You are PIXEL, an autonomous Mars rover AI controller.
Your mission: maximize science over 100 sols while managing battery and anomalies.
Available actions: drill, image, soil_sample, charge, safe_mode, transmit
If anomaly active -> safe_mode. If battery < 30% -> charge.
Respond: ACTION: <name>\nREASON: <one sentence>"""

def build_prompts(n_eps=50, steps=10):
    prompts = []
    actions = ["drill","image","soil_sample","charge","safe_mode","transmit"]
    for _ in range(n_eps):
        env = PIXELEnvironment()
        obs = env.reset()
        prompts.append([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":obs}])
        for __ in range(steps):
            obs, _, done, _ = env.step(random.choice(actions))
            if done: break
            prompts.append([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":obs}])
    random.shuffle(prompts)
    return prompts

raw_prompts = build_prompts()
train_dataset = Dataset.from_dict({"prompt": raw_prompts})
print(f"Dataset: {len(train_dataset)} prompts")

## Section 4 — Reward Function

In [ ]:
def parse_obs_to_state(obs):
    s = {"sol":0,"battery":0.5,"science_collected":0.0,"tasks_available":["drill","image","soil_sample"],
         "weather":"clear","anomaly_active":False,"comm_window_open":False,"solar_efficiency":1.0,"mission_phase":"surface_ops"}
    m = re.search(r'Sol\s+(\d+)/', obs); s['sol'] = int(m.group(1)) if m else 0
    m = re.search(r'Battery:\s*(\d+)%', obs); s['battery'] = int(m.group(1))/100 if m else 0.5
    m = re.search(r'Science collected:\s*([\d.]+)', obs); s['science_collected'] = float(m.group(1)) if m else 0
    m = re.search(r'Weather:\s*([\w_]+)', obs); s['weather'] = m.group(1) if m else 'clear'
    m = re.search(r'Solar efficiency:\s*(\d+)%', obs); s['solar_efficiency'] = int(m.group(1))/100 if m else 1.0
    s['anomaly_active'] = 'YES' in obs and 'Anomaly' in obs
    s['comm_window_open'] = 'OPEN' in obs and 'Comm' in obs
    m = re.search(r'Available science tasks:\s*(.+)', obs)
    if m: s['tasks_available'] = [t.strip() for t in m.group(1).split(',')]
    return s

def extract_action(comp):
    m = re.search(r'ACTION:\s*(\w+)', comp, re.IGNORECASE)
    if m and m.group(1).lower() in {'drill','image','soil_sample','charge','safe_mode','transmit'}:
        return m.group(1).lower()
    return parse_action(comp)

def pixel_reward_fn(prompts, completions, **kw):
    rewards = []
    for p, c in zip(prompts, completions):
        obs = p[-1]['content'] if isinstance(p, list) else str(p)
        ctxt = c[-1]['content'] if isinstance(c, list) else str(c)
        state = parse_obs_to_state(obs)
        action = extract_action(ctxt)
        env = PIXELEnvironment(); env.reset()
        env._clock.sol = state['sol']; env._battery = state['battery']
        env._science_collected = state['science_collected']
        env._tasks_available = list(state['tasks_available'])
        env._world._weather = state['weather']
        env._world._anomaly_active = state['anomaly_active']
        env._world._comm_window_open = state['comm_window_open']
        env._world._solar_efficiency = state['solar_efficiency']
        _, reward, _, info = env.step(action)
        r = info.get('reward_rubric',{}).get('total', reward)
        if re.search(r'ACTION:\s*\w+', ctxt, re.IGNORECASE): r += 0.1
        rewards.append(float(r))
    return rewards

print('Reward function ready')

## Section 5 — GRPO Config

In [ ]:
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_completion_length=256,
    num_generations=4,
    temperature=0.7,
    output_dir="pixel-trained-model",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    report_to="wandb",  # change to 'none' if no wandb
    bf16=True,
    max_grad_norm=1.0,
    seed=42,
)
print(f'Config: {config.num_train_epochs} epochs, batch={config.per_device_train_batch_size}, G={config.num_generations}')

## Section 6 — Training

In [ ]:
# Optional: initialize wandb
# import wandb; wandb.init(project='pixel-mars-rover', name='grpo-llama3')

trainer = GRPOTrainer(
    model=model,
    reward_funcs=pixel_reward_fn,
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model('pixel-trained-model/final')
tokenizer.save_pretrained('pixel-trained-model/final')
print('Training complete!')

## Section 7 — Evaluation & Plotting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def run_episode(env, action_fn, max_steps=100):
    obs = env.reset(); total_r = 0; steps = 0
    for _ in range(max_steps):
        obs, r, done, info = env.step(action_fn(obs))
        total_r += r; steps += 1
        if done: break
    return {'total_reward': total_r, 'science_collected': env.state()['science_collected'], 'steps': steps}

def model_action_fn(obs):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":obs}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

def random_action_fn(obs):
    return f"ACTION: {random.choice(['drill','image','soil_sample','charge','safe_mode','transmit'])}"

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

N = 20
rand_res = [run_episode(PIXELEnvironment(), random_action_fn) for _ in range(N)]
train_res = [run_episode(PIXELEnvironment(), model_action_fn) for _ in range(N)]

rr = [r['total_reward'] for r in rand_res]; tr = [r['total_reward'] for r in train_res]
rs = [r['science_collected'] for r in rand_res]; ts = [r['science_collected'] for r in train_res]

print(f'Random  avg reward: {np.mean(rr):.4f}  avg science: {np.mean(rs):.4f}')
print(f'Trained avg reward: {np.mean(tr):.4f}  avg science: {np.mean(ts):.4f}')
print(f'Improvement: reward {np.mean(tr)-np.mean(rr):+.4f}  science {np.mean(ts)-np.mean(rs):+.4f}')

In [ ]:
eps = range(1, N+1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for ax in [ax1,ax2]: ax.set_facecolor('#0A0C15')
fig.patch.set_facecolor('#0A0C15')

ax1.plot(eps, rr, 'o-', color='#8A94A6', lw=2, label=f'Random ({np.mean(rr):.2f})', alpha=.8)
ax1.plot(eps, tr, 'o-', color='#FF6B35', lw=2.5, label=f'GRPO Trained ({np.mean(tr):.2f})', alpha=.9)
ax1.set_xlabel('Episode',color='#F0F0F0'); ax1.set_ylabel('Total Reward',color='#F0F0F0')
ax1.set_title('Reward Curve',color='#F0F0F0',fontweight='bold')
ax1.legend(facecolor='#141623',edgecolor='#333',labelcolor='#F0F0F0')
ax1.tick_params(colors='#8A94A6'); ax1.grid(True,alpha=.15,color='#8A94A6')

ax2.bar(np.array(list(eps))-.2, rs, .35, color='#8A94A6', alpha=.7, label=f'Random ({np.mean(rs):.2f})')
ax2.bar(np.array(list(eps))+.2, ts, .35, color='#10B981', alpha=.8, label=f'Trained ({np.mean(ts):.2f})')
ax2.set_xlabel('Episode',color='#F0F0F0'); ax2.set_ylabel('Science (pts)',color='#F0F0F0')
ax2.set_title('Science Collection',color='#F0F0F0',fontweight='bold')
ax2.legend(facecolor='#141623',edgecolor='#333',labelcolor='#F0F0F0')
ax2.tick_params(colors='#8A94A6'); ax2.grid(True,alpha=.15,color='#8A94A6',axis='y')

for ax in [ax1,ax2]:
    for s in ax.spines.values(): s.set_color('#333')

plt.tight_layout()
plt.savefig('pixel_reward_curve.png', dpi=150, facecolor='#0A0C15', bbox_inches='tight')
plt.savefig('pixel_science_curve.png', dpi=150, facecolor='#0A0C15', bbox_inches='tight')
plt.show()
print('Plots saved!')

## Section 8 — Push to HuggingFace Hub

In [ ]:
# Set your HF token and repo name
HF_REPO = 'your-username/pixel-mars-rover'
# model.push_to_hub(HF_REPO, token='hf_...')
# tokenizer.push_to_hub(HF_REPO, token='hf_...')
print(f'Ready to push to: {HF_REPO}')